# 03 - Model Training

## Objective

This notebook prepares the cleaned hotel booking dataset for machine learning and trains a Logistic Regression model to predict booking cancellations.

The target variable is `is_canceled`, where:

- `0` = Not canceled
- `1` = Canceled

The main tasks in this notebook include:

- Load the cleaned modeling dataset.
- Define the target variable and feature variables.
- Remove columns that are not suitable for direct modeling.
- Identify numerical and categorical features.
- Split the dataset into training and testing sets.
- Apply scaling to numerical variables.
- Apply One-Hot Encoding to categorical variables.
- Create a preprocessing pipeline using Scikit-Learn.
- Transform the training and testing datasets.
- Train a Logistic Regression model.
- Evaluate model performance using classification metrics.
- Interpret Logistic Regression coefficients.
- Review model coefficients to interpret important cancellation-related features.

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


## 1. Load the Cleaned Dataset

The cleaned modeling dataset generated from the previous cleaning notebook is loaded in this step.

This dataset has already been cleaned and potential data leakage columns have been removed.

In [20]:
df = pd.read_csv("../Data/hotel_bookings_cleaned_model.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (117396, 36)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,has_agent,has_company,arrival_month_num,arrival_date,total_guests,total_nights
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,0,0,7,2015-07-01,1,1
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,0,7,2015-07-01,1,1
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,1,0,7,2015-07-01,2,2
3,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,1,0,7,2015-07-01,2,2
4,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,Transient,107.0,0,0,0,0,7,2015-07-01,2,2


## 2. Understand the Target Variable

The target variable for this project is `is_canceled`.

This is a binary classification problem:

- `0` means the booking was not canceled.
- `1` means the booking was canceled.

Before preprocessing, the class distribution is checked to understand whether the target variable is balanced or imbalanced.

In [21]:
print("Target variable counts:")
print(df["is_canceled"].value_counts())

print("\nTarget variable distribution:")
print(df["is_canceled"].value_counts(normalize=True))

Target variable counts:
is_canceled
0    73646
1    43750
Name: count, dtype: int64

Target variable distribution:
is_canceled
0    0.62733
1    0.37267
Name: proportion, dtype: float64


## 3. Define Features and Target

The target variable `is_canceled` is separated from the feature variables.

Some columns are removed from the feature matrix because they are not suitable for direct machine learning input.

In this step:

- `is_canceled` is used as the target variable.
- `arrival_date` is removed because machine learning models cannot directly use datetime values without further feature engineering.
- `arrival_date_month` is removed because a numerical version of the month already exists in the dataset.

The remaining columns are used as features for model training.

In [22]:
# Define target variable
y = df["is_canceled"]

# Define columns to remove from features
columns_to_drop = [
    "arrival_date",
    "arrival_date_month"
]

# Drop only columns that exist in the dataset
existing_columns_to_drop = [
    col for col in columns_to_drop if col in df.columns
]

# Define feature matrix
X = df.drop(columns=existing_columns_to_drop + ["is_canceled"])

print("Columns dropped from features:", existing_columns_to_drop)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Columns dropped from features: ['arrival_date', 'arrival_date_month']
Feature matrix shape: (117396, 33)
Target shape: (117396,)


## 4. Identify Numerical and Categorical Columns

The feature variables are divided into two groups:

- Numerical columns: variables stored as numbers.
- Categorical columns: variables stored as text or categories.

Numerical columns will be scaled using `StandardScaler`.

Categorical columns will be converted into numerical format using `OneHotEncoder`, because machine learning models cannot directly understand text values.

In [23]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Number of numerical columns:", len(numerical_cols))
print(numerical_cols)

print("\nNumber of categorical columns:", len(categorical_cols))
print(categorical_cols)

Number of numerical columns: 24
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'has_agent', 'has_company', 'arrival_month_num', 'total_guests', 'total_nights']

Number of categorical columns: 9
['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']


## 5. Split the Dataset into Training and Testing Sets

The dataset is split into training and testing sets.

- 80% of the data is used for training.
- 20% of the data is used for testing.

The parameter `stratify=y` is used to preserve the original class distribution of the target variable in both training and testing sets.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (93916, 33)
X_test shape: (23480, 33)

y_train shape: (93916,)
y_test shape: (23480,)

Training target distribution:
is_canceled
0    0.627327
1    0.372673
Name: proportion, dtype: float64

Testing target distribution:
is_canceled
0    0.627342
1    0.372658
Name: proportion, dtype: float64


## 6. Create Preprocessing Pipelines

Two preprocessing pipelines are created:

- Numerical pipeline: applies `StandardScaler` to numerical variables.
- Categorical pipeline: applies `OneHotEncoder` to categorical variables.

`StandardScaler` standardizes numerical values so that they have a similar scale.

`OneHotEncoder` converts categorical variables into numerical dummy variables. The parameter `handle_unknown="ignore"` prevents errors if unseen categories appear in the test set or future data.

In [25]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

## 7. Combine Preprocessing Steps with ColumnTransformer

`ColumnTransformer` is used to apply different preprocessing steps to different types of columns.

In this project:

- Numerical columns are scaled.
- Categorical columns are one-hot encoded.

This makes the preprocessing process more organized and suitable for future model training.

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'children',
                                  'babies', 'is_repeated_guest',
                                  'previous_cancellations',
                                  'previous_bookings_not_canceled',
                                  'booking_changes', 'ag...
                                  'required_car_parking_spaces',
                                  'total_of_special_requests', 'has_agent',
                                  'has_company', 'arrival_month_num',
                                  'total_guests', 'total_nights']),
             

## 8. Fit the Preprocessing Pipeline on Training Data

The preprocessing pipeline is fitted only on the training data.

This is important because fitting on the full dataset before train-test split would allow information from the test set to influence the preprocessing step.

Fitting only on training data helps avoid data leakage.

In [27]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 9. Transform Training and Testing Datasets

After fitting the preprocessing pipeline on the training data, both the training and testing feature sets are transformed.

The result is a machine-learning-ready numerical matrix that can be used for model training and evaluation.

In [28]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (93916, 242)
Processed X_test shape: (23480, 242)


## 10. View Transformed Feature Names

One-Hot Encoding creates new columns from categorical variables.

This step extracts and displays the final feature names after preprocessing. This helps make the transformed dataset easier to understand and inspect.

In [29]:
encoded_feature_names = (
    preprocessor.named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(categorical_cols)
)

all_feature_names = numerical_cols + list(encoded_feature_names)

print("Total processed features:", len(all_feature_names))

print("\nFirst 30 feature names:")
print(all_feature_names[:30])

Total processed features: 242

First 30 feature names:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'has_agent', 'has_company', 'arrival_month_num', 'total_guests', 'total_nights', 'hotel_City Hotel', 'hotel_Resort Hotel', 'meal_BB', 'meal_FB', 'meal_HB', 'meal_SC']


## 11. Convert Processed Matrices into DataFrames

The processed training and testing matrices are converted into pandas DataFrames.

This step makes the processed data easier to inspect, save, and use in the modeling stage.

In [30]:
X_train_df = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=all_feature_names
)

X_test_df = pd.DataFrame(
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed,
    columns=all_feature_names
)

print("Processed training DataFrame shape:", X_train_df.shape)
print("Processed testing DataFrame shape:", X_test_df.shape)

X_train_df.head()

Processed training DataFrame shape: (93916, 242)
Processed testing DataFrame shape: (23480, 242)


,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,...,assigned_room_type_I,assigned_room_type_K,assigned_room_type_L,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,-0.941364,-0.217628,1.535554,0.592313,0.072757,-1.329071,-1.486373,-0.250315,-0.078733,5.739465,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.291429,1.195939,-0.152742,0.706128,-0.933132,0.260986,0.250068,-0.250315,-0.078733,-0.174232,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.254071,1.195939,0.361087,-0.545835,-0.933132,-0.799052,0.250068,-0.250315,-0.078733,-0.174232,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.707880,-0.217628,-0.519763,-0.659650,-0.933132,1.321024,0.250068,-0.250315,-0.078733,-0.174232,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.819952,-1.631196,0.728108,-0.659650,1.078645,2.911081,0.250068,-0.250315,-0.078733,-0.174232,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


## 12. Train Logistic Regression Model

After preprocessing, the dataset is ready for machine learning.

Logistic Regression is used as the first baseline model because it is simple, interpretable, and suitable for binary classification problems.

In this project, the target variable is `is_canceled`:

- `0` = Not canceled
- `1` = Canceled

The model learns patterns from the processed training data and predicts whether bookings in the test set are likely to be canceled.

In [31]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [ ]:
# Initialize Logistic Regression model
log_reg_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
log_reg_model.fit(X_train_processed, y_train)

print("Logistic Regression model trained successfully.")

## 13. Make Predictions on the Test Set

After training, the model is used to make predictions on the test set.

Two types of predictions are generated:

- Predicted class labels: `0` or `1`
- Predicted probabilities: the probability that a booking will be canceled

In [ ]:
# Predict class labels
y_pred = log_reg_model.predict(X_test_processed)

# Predict cancellation probabilities
y_pred_proba = log_reg_model.predict_proba(X_test_processed)[:, 1]

print("First 20 predictions:")
print(y_pred[:20])

print("\nFirst 20 predicted cancellation probabilities:")
print(y_pred_proba[:20])

First 20 predictions:
[1 0 0 0 0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0]

First 20 predicted cancellation probabilities:
[6.39263743e-01 4.41466702e-01 2.23093896e-02 3.62822876e-09
 6.21611726e-02 3.82180760e-01 1.29666264e-01 7.47530448e-01
 1.87554920e-01 9.94741860e-01 9.98347341e-01 5.31823325e-02
 2.79023200e-02 1.60749280e-10 9.99236690e-01 3.42662619e-01
 2.52627431e-01 2.32376053e-03 1.09060866e-01 4.97704405e-01]


## 14. Evaluate the Logistic Regression Model

The model is evaluated using several classification metrics:

- Accuracy: the percentage of correct predictions.
- Precision: among bookings predicted as canceled, how many were actually canceled.
- Recall: among actual canceled bookings, how many were correctly detected.
- F1-score: the balance between precision and recall.
- ROC-AUC: the model's ability to distinguish between canceled and non-canceled bookings.
- Confusion Matrix: a table showing correct and incorrect predictions.

For this project, accuracy alone is not enough because the business objective is to detect bookings that are likely to be canceled.

In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("Logistic Regression Evaluation")
print("Accuracy:", round(accuracy, 4))
print("ROC-AUC:", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

Logistic Regression Evaluation
Accuracy: 0.8181
ROC-AUC: 0.8993

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86     14730
           1       0.81      0.67      0.73      8750

    accuracy                           0.82     23480
   macro avg       0.82      0.79      0.80     23480
weighted avg       0.82      0.82      0.81     23480


Confusion Matrix:
[[13364  1366]
 [ 2906  5844]]


## 15. Interpret Logistic Regression Coefficients

Logistic Regression coefficients help explain how each feature affects the probability of booking cancellation.

Interpretation:

- A positive coefficient means the feature increases the probability of cancellation.
- A negative coefficient means the feature decreases the probability of cancellation.

Features with larger absolute coefficient values have stronger influence on the model prediction.

For example:

- A high positive coefficient for `previous_cancellations` suggests that customers who canceled before are more likely to cancel again.

- A positive coefficient for `deposit_type_Non Refund` may indicate that customers choosing non-refundable deposits are associated with higher cancellation behavior in this dataset.

- A negative coefficient for `required_car_parking_spaces` suggests that customers requesting parking spaces are less likely to cancel because they may have stronger travel commitment.

This interpretation helps businesses better understand customer behavior and identify important factors related to booking cancellation.

In [ ]:
coefficients = pd.DataFrame({
    "Feature": all_feature_names,
    "Coefficient": log_reg_model.coef_[0]
})

coefficients = coefficients.sort_values(
    by="Coefficient",
    ascending=False
)

print("Top 10 features increasing cancellation probability:")
display(coefficients.head(10))

print("\nTop 10 features decreasing cancellation probability:")
display(coefficients.tail(10))

Top 10 features increasing cancellation probability:


,Feature,Coefficient
236,deposit_type_Non Refund,2.612999
34,country_ARE,2.450087
95,country_HKG,2.134781
147,country_NGA,2.090492
10,previous_cancellations,1.878179
224,assigned_room_type_A,1.847703
157,country_PHL,1.842176
31,country_AGO,1.453759
221,reserved_room_type_G,1.450241
99,country_IDN,1.433979



Top 10 features decreasing cancellation probability:


,Feature,Coefficient
133,country_MEX,-1.121130
125,country_LTU,-1.227917
105,country_ISL,-1.323990
230,assigned_room_type_G,-1.482736
215,reserved_room_type_A,-1.521415
174,country_SRB,-1.626497
152,country_NZL,-1.660868
232,assigned_room_type_I,-1.740826
235,deposit_type_No Deposit,-1.990632
17,required_car_parking_spaces,-4.752556


### Interpretation of Logistic Regression Coefficients

The coefficient table shows which features increase or decrease the predicted probability of booking cancellation.

From the model results, `deposit_type_Non Refund` has the largest positive coefficient. This means that bookings with a non-refundable deposit type are strongly associated with a higher probability of cancellation in this dataset.

`previous_cancellations` also has a positive coefficient, suggesting that customers who canceled bookings in the past are more likely to cancel again.

Several country-related dummy variables also appear among the top positive or negative coefficients. These features should be interpreted carefully because they may reflect specific booking patterns in the dataset rather than universal business rules.

On the other hand, `required_car_parking_spaces` has a strong negative coefficient. This suggests that guests who request parking spaces are less likely to cancel, possibly because they have stronger travel intention.

`deposit_type_No Deposit` also has a negative coefficient, meaning that compared with other deposit categories, this group is associated with a lower cancellation probability in the trained model.

Overall, the coefficient analysis confirms that deposit policy, previous cancellation behavior, parking requests, room type, and country-related patterns are important factors in the Logistic Regression model.

## Conclusion

In this notebook, the cleaned hotel booking dataset was prepared for machine learning and used to train a Logistic Regression model.

The target variable `is_canceled` was separated from the feature variables, and unnecessary columns were removed before modeling. Numerical and categorical features were identified, then processed using a Scikit-Learn preprocessing pipeline.

Numerical variables were scaled using `StandardScaler`, while categorical variables were transformed using `OneHotEncoder`. The dataset was split into training and testing sets, and the preprocessing pipeline was fitted only on the training data to avoid data leakage.

A Logistic Regression model was trained on the processed training data and evaluated on the test data. The model performance was assessed using accuracy, ROC-AUC, classification report, and confusion matrix.

The Logistic Regression model achieved an accuracy of 0.8181 and a ROC-AUC score of 0.8993, showing strong baseline performance.

Finally, the Logistic Regression coefficients were examined to understand which features were associated with higher or lower cancellation probability.
